# Phase 2 — Qwen3-1.7B + Qwen3-8B + LFM2.5-1.2B + Llama3.1-8B (QLoRA)

**GPU 0**

| Model | Size | JSON valid % | Notes |
|-------|------|-------------|-------|
| `Qwen/Qwen3-1.7B` | 1.7B | 100% | |
| `Qwen/Qwen3-8B` | 8B | 95% | |
| `LiquidAI/LFM2.5-1.2B-Instruct` | 1.2B | 100% | trust_remote_code=True |
| `meta-llama/Meta-Llama-3.1-8B-Instruct` | 8B | 100% | |

| | |
|---|---|
| **Data** | `./` (same directory) |
| **Output** | `./adapters/` |
| **Log** | `logs/phase2_group1.log` |

In [ ]:
import os, subprocess

# ── SINGLE GPU — pick one free GPU, leave others for colleagues ──────────────
GPU_INDEX = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_INDEX   # hard-set, not setdefault

# ── CPU THREAD LIMITS — leave cores for other users ──────────────────────────
# Cap training threads — leave cores for other users.
N_THREADS = '8'
os.environ['OMP_NUM_THREADS']        = N_THREADS   # OpenMP (PyTorch CPU ops)
os.environ['MKL_NUM_THREADS']        = N_THREADS   # Intel MKL
os.environ['OPENBLAS_NUM_THREADS']   = N_THREADS   # OpenBLAS fallback
os.environ['NUMEXPR_NUM_THREADS']    = N_THREADS   # numexpr
os.environ['TOKENIZERS_PARALLELISM'] = 'false'     # HF tokenizer — no fork
os.environ['USE_TF']                 = '0'
os.environ['TRANSFORMERS_NO_TF']     = '1'

# Apply torch inter/intra-op threads AFTER torch import
import torch
torch.set_num_threads(8)          # intra-op parallelism
torch.set_num_interop_threads(4)  # inter-op parallelism (set once before use)

# ── VERIFY GPU STATE ─────────────────────────────────────────────────────────
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,memory.used,memory.free',
     '--format=csv,noheader', f'--id={GPU_INDEX}'],
    capture_output=True, text=True
)
idx, used, free = result.stdout.strip().split(', ')
used_mib = int(used.replace(' MiB', ''))
if used_mib > 1000:
    print(f'WARNING: GPU {GPU_INDEX} already has {used}  used — consider switching GPU_INDEX')
else:
    print(f'GPU {GPU_INDEX}: {used} used / {free} free — OK')

print(f'CUDA_VISIBLE_DEVICES = {GPU_INDEX}')
print(f'CPU threads: OMP/MKL/OpenBLAS = {N_THREADS}, torch intra=8 inter=4')
print(f'TOKENIZERS_PARALLELISM = false')

# ── HF TOKEN ─────────────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF_TOKEN: set')
else:
    print('HF_TOKEN: not set — run: export HF_TOKEN=hf_...')


In [ ]:
# Version check — no install needed (using existing environment)
import importlib.metadata as meta
for pkg in ['transformers', 'peft', 'bitsandbytes', 'accelerate']:
    try:
        print(f'  {pkg}: {meta.version(pkg)}')
    except meta.PackageNotFoundError:
        print(f'  {pkg}: NOT INSTALLED')


In [ ]:
import os
from pathlib import Path
for f in ['features.csv', 'patient_notes.csv', 'train.csv']:
    print(f'  {f}: {chr(10003) if os.path.exists(f) else chr(10007) + " MISSING"}')
aug_candidates = [Path('phase1_output/augmented_train.csv'), Path('augmented_train.csv')]
aug_path = next((p for p in aug_candidates if p.exists()), None)
aug_status = f'{chr(10003)} found at {aug_path}' if aug_path else 'WARNING: not found — will train on train.csv only'
print(f'  augmented_train.csv: {aug_status}')


## Configuration

| Key | Default | When to change |
|-----|---------|----------------|
| `PER_DEVICE_BATCH_SIZE` | `8` | Lower to `4` if OOM |
| `GRADIENT_ACCUMULATION` | `2` | Effective batch = 8×2 = 16 |
| `NUM_TRAIN_EPOCHS` | `3` | Increase for better convergence |
| `MAX_STEPS` | `-1` | Set to `200` for quick test run |
| `AUG_SAMPLE_RATIO` | `1.0` | Lower to `0.1` to subsample augmented data |
| `GPU_INDEX` | `'0'` | Change to free GPU index |

In [ ]:
from transformers import TrainerCallback
import time

class SafeEvalGenerationCallback(TrainerCallback):
    """
    Injects task-specific generation metrics into Trainer's eval pipeline
    without mutating internal state or blocking training.
    """
    def __init__(self, val_dataset, tokenizer, model_spec, cfg, max_eval_samples: int = 64):
        self.val_dataset = val_dataset
        self.tokenizer = tokenizer
        self.model_spec = model_spec
        self.cfg = cfg
        self.max_eval_samples = max_eval_samples

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return control
            
        # Fail-safe: ensure model & tokenizer are accessible via Trainer kwargs
        model = kwargs.get("model")
        if model is None:
            return control
            
        # Subsample to prevent eval timeout / OOM
        eval_subset = self.val_dataset
        if len(eval_subset) > self.max_eval_samples:
            eval_subset = eval_subset.select(range(self.max_eval_samples))
            
        try:
            gen_metrics = run_validation_generation(
                model,
                self.tokenizer,
                self.model_spec,
                eval_subset,
                self.cfg
            )
            # Prefix to avoid collision with default Trainer metrics
            prefixed = {f"eval_gen_{k}": v for k, v in gen_metrics.items()}
            metrics.update(prefixed)
        except Exception as e:
            metrics["eval_gen_failure"] = 1.0
            metrics["eval_gen_error"] = str(e)
            
        return control

class ResourceAndStabilityCallback(TrainerCallback):
    """Logs GPU memory, step duration, and LoRA gradient norms at each logging step."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or not state.is_world_process_zero:
            return control
            
        if torch.cuda.is_available():
            try:
                logs["gpu_mem_alloc_gb"] = torch.cuda.memory_allocated(0) / 1024**3
                logs["gpu_mem_reserved_gb"] = torch.cuda.memory_reserved(0) / 1024**3
            except Exception:
                pass
                
        model = kwargs.get("model")
        if model is not None and hasattr(model, "base_model"):
            try:
                lora_params = [p for n, p in model.named_parameters() if "lora_" in n and p.requires_grad]
                if lora_params:
                    lora_grad_norm = torch.norm(torch.stack([torch.norm(p.grad.detach()) for p in lora_params if p.grad is not None]))
                    logs["lora_grad_norm"] = lora_grad_norm.item()
            except Exception:
                pass
        return control

In [ ]:
import ast, gc, json, logging, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import GroupKFold
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq,
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed,
)

CONFIG = {
    "DATA_DIR":              Path("."),
    "ADAPTER_ROOT":          Path("./adapters"),
    "AUG_CSV":               Path("./augmented_train.csv"),
    "SEED":                  42,
    "AUG_SAMPLE_RATIO":      0.5,   # reduced from 1.0 — ablate distribution shift
    "N_FOLDS":               5,
    "VAL_FOLD":              4,
    "LORA_R":                8,     # reduced from 16 — less capacity → less memorization
    "LORA_ALPHA":            16,    # keep alpha/r = 2.0
    "LORA_DROPOUT":          0.1,   # increased from 0.05 — stronger adapter regularization
    "LORA_TARGET_MODULES":   ["q_proj", "k_proj", "v_proj", "o_proj"],  # attention-only
    "VAL_GENERATION_N":      128,
    "PER_DEVICE_BATCH_SIZE": 8,
    "GRADIENT_ACCUMULATION": 2,       # effective batch = 8x2 = 16
    "LEARNING_RATE":         1e-4,  # reduced from 2e-4 — slower memorization
    "NUM_TRAIN_EPOCHS":      3,
    "MAX_SEQ_LENGTH":        1024,
    "WARMUP_RATIO":          0.05,
    "LR_SCHEDULER":          "cosine",
    "WEIGHT_DECAY":          0.05,  # increased from 0.01 — stronger L2 pressure
    "LOGGING_STEPS":         50,
    "SAVE_STEPS":            500,
    "EVAL_STEPS":            500,
    "EVAL_STRATEGY":         "steps",
    "SAVE_TOTAL_LIMIT":      2,     # keep best + latest
    "MAX_STEPS":             -1,
}

# Per-model seeds for ensemble diversity (Fix E)
MODEL_SEEDS = {
    "meta-llama/Llama-3.1-8B-Instruct":    42,
    "Qwen/Qwen3-1.7B":                      43,
    "Qwen/Qwen3-8B":                        44,
    "LiquidAI/LFM2.5-1.2B-Instruct":       45,
}

MODEL_REGISTRY = [
    {
        "name":              "llama3_1_8b",
        "model_id":          "meta-llama/Llama-3.1-8B-Instruct",
        "model_class":       "causal_lm",
        "compute_dtype":     torch.bfloat16,
        "fp16":              False,
        "bf16":              True,
        "adapter_dir":       Path("./adapters/llama3_1_8b_adapter"),
        "enable_thinking":   False,
        "trust_remote_code": False,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    {
        "name":              "qwen3_1_7b",
        "model_id":          "Qwen/Qwen3-1.7B",
        "model_class":       "causal_lm",
        "compute_dtype":     torch.bfloat16,
        "fp16":              False,
        "bf16":              True,
        "adapter_dir":       Path("./adapters/qwen3_1_7b_adapter"),
        "enable_thinking":   False,
        "trust_remote_code": False,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    {
        "name":              "qwen3_8b",
        "model_id":          "Qwen/Qwen3-8B",
        "model_class":       "causal_lm",
        "compute_dtype":     torch.bfloat16,
        "fp16":              False,
        "bf16":              True,
        "adapter_dir":       Path("./adapters/qwen3_8b_adapter"),
        "enable_thinking":   False,
        "trust_remote_code": False,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    {
        "name":              "lfm2_5_1_2b",
        "model_id":          "LiquidAI/LFM2.5-1.2B-Instruct",
        "model_class":       "causal_lm",
        "compute_dtype":     torch.bfloat16,
        "fp16":              False,
        "bf16":              True,
        "adapter_dir":       Path("./adapters/lfm2_5_1_2b_adapter"),
        "enable_thinking":   None,
        "trust_remote_code": True,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj"],
    },
]

SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation.\n"
    '{"spans": ["exact text 1", "exact text 2"]}'
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  [%(levelname)s]  %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print('CONFIG, MODEL_REGISTRY, SYSTEM_PROMPT loaded')
print(f'  Group 1 models: {[m["name"] for m in MODEL_REGISTRY]}')
print(f'  Data dir: {CONFIG["DATA_DIR"]}')


## Section 1 — Data Loading & Preparation

Loads `train.csv` and (optionally) `augmented_train.csv`, merges them, then joins with
`patient_notes.csv` and `features.csv` to get the full `pn_history` and `feature_text`.

The `assistant_target` column is the JSON string the model should learn to produce:
`{"spans": ["exact text"]}` for positive examples, `{"spans": []}` for negatives.

In [ ]:
def safe_parse_list(val) -> list:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []


def build_assistant_response(annotation_list: list) -> str:
    clean_spans = [s.strip() for s in annotation_list if isinstance(s, str) and s.strip()]
    return json.dumps({"spans": clean_spans}, ensure_ascii=False)


def load_and_merge_data(cfg: dict) -> pd.DataFrame:
    data_dir = cfg["DATA_DIR"]
    log.info("Loading CSVs ...")
    train_df    = pd.read_csv(data_dir / "train.csv")
    pn_df       = pd.read_csv(data_dir / "patient_notes.csv")
    features_df = pd.read_csv(data_dir / "features.csv")
    train_df["annotation"] = train_df["annotation"].apply(safe_parse_list)

    aug_path = cfg.get("AUG_CSV", data_dir / "augmented_train.csv")
    if not aug_path.exists() and (data_dir / "augmented_train.csv").exists():
        aug_path = data_dir / "augmented_train.csv"

    if aug_path.exists():
        aug_df = pd.read_csv(aug_path)
        aug_df["annotation"] = aug_df["annotation"].apply(safe_parse_list)
        ratio = cfg["AUG_SAMPLE_RATIO"]
        if ratio < 1.0:
            n_sample = max(1, int(len(aug_df) * ratio))
            aug_df   = aug_df.sample(n=n_sample, random_state=cfg["SEED"])
            log.info(f"  Augmented data sampled: {n_sample} / {len(pd.read_csv(aug_path))} ({100*ratio:.0f}%)")
        combined = pd.concat([train_df, aug_df], ignore_index=True)
        log.info(f"  Combined: {len(train_df)} (train) + {len(aug_df)} (augmented) = {len(combined)}")
    else:
        log.warning("augmented_train.csv not found — training on train.csv only.")
        combined = train_df.copy()

    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = features_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()

    combined["pn_history"]       = combined["pn_num"].map(pn_map).fillna("")
    combined["feature_text"]     = combined.apply(lambda r: feat_map.get((r["case_num"], r["feature_num"]), ""), axis=1)
    combined["assistant_target"] = combined["annotation"].apply(build_assistant_response)

    span_counts = combined["annotation"].map(len)
    log.info(f"  Targets: {(span_counts > 0).sum()} positive rows, {(span_counts == 0).sum()} empty rows")
    log.info(f"  Span count distribution: {dict(sorted(Counter(span_counts).items()))}")

    before   = len(combined)
    combined = combined[combined["pn_history"].str.strip().ne("") & combined["feature_text"].str.strip().ne("")].reset_index(drop=True)
    log.info(f"  Dropped {before - len(combined)} empty rows. Remaining: {len(combined)}")
    return combined[["pn_num", "case_num", "feature_num", "pn_history", "feature_text", "annotation", "assistant_target"]]

print("Section 1 defined")



## Section 2 — GroupKFold Split

Splits by `case_num` (10 unique cases → 5 folds of 2 cases each). This ensures no patient
from the validation set appears in training — the clinically correct way to avoid leakage,
since notes from the same case have correlated vocabulary.

In [ ]:
def make_train_val_datasets(df: pd.DataFrame, cfg: dict) -> tuple:
    gkf    = GroupKFold(n_splits=cfg["N_FOLDS"])
    groups = df["case_num"].values
    X      = np.arange(len(df))

    train_idx = val_idx = None
    for fold, (tr_idx, vl_idx) in enumerate(gkf.split(X, groups=groups)):
        if fold == cfg["VAL_FOLD"]:
            train_idx, val_idx = tr_idx, vl_idx
            break

    if train_idx is None:
        all_idx   = set(range(len(df)))
        train_idx = np.array(sorted(all_idx - set(val_idx)))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)
    log.info(
        f"GroupKFold: train={len(train_df)} rows "
        f"(cases {sorted(train_df['case_num'].unique())}) | "
        f"val={len(val_df)} rows (cases {sorted(val_df['case_num'].unique())})"
    )
    return Dataset.from_pandas(train_df), Dataset.from_pandas(val_df)

print("✓ Section 2: make_train_val_datasets defined")

## Section 3 — Prompt Formatting & Assistant-Only Labels

Builds the same chat prompt used at inference, then masks prompt tokens with `-100` so loss is computed only on the assistant JSON response.


In [ ]:
def build_messages(pn_history: str, feature_text: str, assistant_target: str = None) -> list:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Note: \"{(pn_history or '').strip()}\"\n"
            f"Feature: {feature_text or ''}"
        )},
    ]
    if assistant_target is not None:
        messages.append({"role": "assistant", "content": assistant_target or '{"spans": []}'})
    return messages


def render_prompt(tokenizer, model_spec: dict, pn_history: str, feature_text: str) -> str:
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    if model_spec.get("enable_thinking") is not None:
        kwargs["enable_thinking"] = model_spec["enable_thinking"]
    return tokenizer.apply_chat_template(build_messages(pn_history, feature_text), **kwargs)


def render_training_text(tokenizer, model_spec: dict, pn_history: str, feature_text: str, assistant_target: str) -> str:
    kwargs = dict(tokenize=False, add_generation_prompt=False)
    if model_spec.get("enable_thinking") is not None:
        kwargs["enable_thinking"] = model_spec["enable_thinking"]
    return tokenizer.apply_chat_template(build_messages(pn_history, feature_text, assistant_target), **kwargs)


def tokenize_for_assistant_loss(example: dict, tokenizer, model_spec: dict, cfg: dict) -> dict:
    prompt_text = render_prompt(tokenizer, model_spec, example["pn_history"], example["feature_text"])
    full_text = render_training_text(tokenizer, model_spec, example["pn_history"], example["feature_text"], example["assistant_target"])
    tokenized = tokenizer(full_text, truncation=True, max_length=cfg["MAX_SEQ_LENGTH"], add_special_tokens=False)
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=cfg["MAX_SEQ_LENGTH"], add_special_tokens=False)["input_ids"]
    labels = list(tokenized["input_ids"])
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    if all(label == -100 for label in labels):
        raise ValueError("All labels were masked; increase MAX_SEQ_LENGTH or inspect chat template.")
    tokenized["labels"] = labels
    return tokenized


def prepare_tokenized_datasets(train_dataset: Dataset, val_dataset: Dataset, tokenizer, model_spec: dict, cfg: dict) -> tuple:
    def _map(example):
        return tokenize_for_assistant_loss(example, tokenizer, model_spec, cfg)
    train_tok = train_dataset.map(_map, remove_columns=train_dataset.column_names)
    val_tok = val_dataset.map(_map, remove_columns=val_dataset.column_names)
    return train_tok, val_tok


def validate_token_lengths(df: pd.DataFrame, tokenizer, model_spec: dict, cfg: dict) -> None:
    lengths = []
    for row in df.itertuples(index=False):
        text = render_training_text(tokenizer, model_spec, row.pn_history, row.feature_text, row.assistant_target)
        lengths.append(len(tokenizer(text, add_special_tokens=False)["input_ids"]))
    arr = np.array(lengths)
    log.info(f"[{model_spec['name']}] token lengths: p50={int(np.percentile(arr, 50))} p90={int(np.percentile(arr, 90))} p99={int(np.percentile(arr, 99))} max={int(arr.max())}")
    too_long = int((arr > cfg["MAX_SEQ_LENGTH"]).sum())
    if too_long:
        raise ValueError(f"{model_spec['name']}: {too_long} rows exceed MAX_SEQ_LENGTH={cfg['MAX_SEQ_LENGTH']}")

print("Section 3 defined")



## Section 4 — Model & Tokenizer Loading

Loads base model in 4-bit NF4 quantization (QLoRA). All models use `AutoModelForCausalLM`.

`device_map={'': 'cuda:0'}` pins model to single GPU controlled by `CUDA_VISIBLE_DEVICES`.

`attn_implementation='eager'` is safe for all models in this registry.

`trust_remote_code=True` required for LiquidAI/LFM2.5 (custom architecture).

In [ ]:
def build_bnb_config(compute_dtype: torch.dtype) -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = 'nf4',
        bnb_4bit_compute_dtype    = compute_dtype,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_storage    = compute_dtype,
    )


def load_model_and_tokenizer(model_spec: dict, bnb_config: BitsAndBytesConfig):
    model_id           = model_spec['model_id']
    trust_remote_code  = model_spec.get('trust_remote_code', False)
    log.info(f'Loading model: {model_id} ...')

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config = bnb_config,
        torch_dtype         = model_spec['compute_dtype'],
        device_map          = {'': 'cuda:0'},  # pin to single GPU (set CUDA_VISIBLE_DEVICES first)
        attn_implementation = 'eager',         # safe default for all causal_lm models
        trust_remote_code   = trust_remote_code,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_id, use_fast=True, trust_remote_code=trust_remote_code
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
        log.info('  pad_token set to eos_token')
    tokenizer.padding_side = 'right'
    log.info(f'  vocab_size={tokenizer.vocab_size}  pad="{tokenizer.pad_token}"')
    return model, tokenizer

print('✓ Section 4: build_bnb_config, load_model_and_tokenizer defined')

## Section 5 — LoRA Adapter Setup

Injects LoRA adapters into all 7 projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`,
`gate_proj`, `up_proj`, `down_proj`) with rank 16.

`prepare_model_for_kbit_training` enables gradient checkpointing and casts LayerNorm to
float32 for stable training on top of 4-bit weights.

In [ ]:
def build_lora_config(cfg: dict, model_spec: dict = None) -> LoraConfig:
    return LoraConfig(
        r              = cfg["LORA_R"],
        lora_alpha     = cfg["LORA_ALPHA"],
        target_modules = (model_spec or {}).get("lora_target_modules") or cfg["LORA_TARGET_MODULES"],
        lora_dropout   = cfg["LORA_DROPOUT"],
        bias           = "none",
        task_type      = TaskType.CAUSAL_LM,
    )


def validate_lora_injection(model, model_spec: dict) -> None:
    target_modules = model_spec.get("lora_target_modules") or CONFIG["LORA_TARGET_MODULES"]
    if isinstance(target_modules, str):
        log.info(f"[{model_spec['name']}] LoRA target regex: {target_modules}")
        return
    adapted = [name for name, module in model.named_modules() if hasattr(module, "lora_A") or hasattr(module, "lora_B")]
    counts = {target: sum(name.endswith(f".{target}") or name == target for name in adapted) for target in target_modules}
    missing = [target for target, count in counts.items() if count == 0]
    log.info(f"[{model_spec['name']}] LoRA module counts: {counts}")
    if missing:
        raise ValueError(f"{model_spec['name']}: LoRA targets not found: {missing}")


def apply_lora(model, lora_config: LoraConfig):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

print("Section 5 defined")



## Section 6 — Training Configuration & Validation Metrics

Uses `Trainer` on pre-tokenized causal-LM examples with explicit assistant-only labels, plus a small generation check on the validation fold.


In [ ]:
def build_training_args(model_spec: dict, adapter_dir: Path, cfg: dict) -> TrainingArguments:
    seed = MODEL_SEEDS.get(model_spec["model_id"], cfg["SEED"])
    return TrainingArguments(
        output_dir                   = str(adapter_dir / "checkpoints"),
        num_train_epochs             = cfg["NUM_TRAIN_EPOCHS"],
        max_steps                    = cfg["MAX_STEPS"],
        per_device_train_batch_size  = cfg["PER_DEVICE_BATCH_SIZE"],
        per_device_eval_batch_size   = cfg["PER_DEVICE_BATCH_SIZE"],
        gradient_accumulation_steps  = cfg["GRADIENT_ACCUMULATION"],
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        learning_rate                = cfg["LEARNING_RATE"],
        weight_decay                 = cfg["WEIGHT_DECAY"],
        warmup_ratio                 = cfg["WARMUP_RATIO"],
        lr_scheduler_type            = cfg["LR_SCHEDULER"],
        optim                        = "adamw_8bit",
        fp16                         = model_spec["fp16"],
        bf16                         = model_spec["bf16"],
        eval_strategy                = cfg["EVAL_STRATEGY"],
        eval_steps                   = cfg["EVAL_STEPS"],
        save_strategy                = "steps",
        save_steps                   = cfg["SAVE_STEPS"],
        save_total_limit             = cfg["SAVE_TOTAL_LIMIT"],
        load_best_model_at_end       = True,
        metric_for_best_model        = "eval_loss",
        greater_is_better            = False,
        logging_steps                = cfg["LOGGING_STEPS"],
        logging_dir                  = str(adapter_dir / "logs"),
        report_to                    = ["tensorboard"],
        seed                         = seed,
        data_seed                    = seed,
        remove_unused_columns        = False,
        dataloader_num_workers       = 0,
    )


def parse_spans(raw_text: str) -> list:
    try:
        parsed = json.loads(raw_text.strip())
    except json.JSONDecodeError:
        start = raw_text.find("{")
        end = raw_text.rfind("}")
        if start < 0 or end <= start:
            return []
        try:
            parsed = json.loads(raw_text[start:end + 1])
        except json.JSONDecodeError:
            return []
    spans = parsed.get("spans", []) if isinstance(parsed, dict) else []
    return [s.strip() for s in spans if isinstance(s, str) and s.strip()]


def spans_to_char_set(note: str, spans: list) -> set:
    chars = set()
    for span in spans:
        start = note.find(span)
        while start != -1:
            chars.update(range(start, start + len(span)))
            start = note.find(span, start + 1)
    return chars


def run_validation_generation(model, tokenizer, model_spec: dict, val_dataset: Dataset, cfg: dict) -> dict:
    n_eval = min(cfg.get("VAL_GENERATION_N", 0), len(val_dataset))
    if n_eval <= 0:
        return {}
    subset = val_dataset.select(range(n_eval))
    model.eval()
    parse_ok = contained_ok = empty_correct = 0
    tp = fp = fn = 0
    for row in subset:
        prompt = render_prompt(tokenizer, model_spec, row["pn_history"], row["feature_text"])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        new_tokens = generated[0, inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
        pred_spans = parse_spans(raw)
        true_spans = row["annotation"]
        parse_ok += int(raw.strip().startswith("{") or bool(pred_spans) or '"spans"' in raw)
        contained_ok += int(all(span in row["pn_history"] for span in pred_spans))
        empty_correct += int((len(true_spans) == 0) == (len(pred_spans) == 0))
        pred_chars = spans_to_char_set(row["pn_history"], pred_spans)
        true_chars = spans_to_char_set(row["pn_history"], true_spans)
        tp += len(pred_chars & true_chars)
        fp += len(pred_chars - true_chars)
        fn += len(true_chars - pred_chars)
    char_f1 = (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else 1.0
    metrics = {
        "val_gen_n": n_eval,
        "val_jsonish_rate": parse_ok / n_eval,
        "val_contained_rate": contained_ok / n_eval,
        "val_empty_accuracy": empty_correct / n_eval,
        "val_char_f1": char_f1,
    }
    log.info(f"[{model_spec['name']}] validation generation metrics: {metrics}")
    return metrics

print("Section 6 defined")


## Section 7 — Training Pipeline (One Model)

Orchestrates the full QLoRA training loop for one SLM:

1. Skip if adapter already saved (safe to re-run)
2. Load quantized model + tokenizer
3. Validate sequence lengths for the actual NBME prompt/target format
4. Inject LoRA adapters and validate target module matches
5. Train with `Trainer` on assistant-response-only labels
6. Save adapter weights only and run a small generation/F1 validation check
7. Aggressive VRAM cleanup before the next model loads


In [ ]:
def train_one_model(model_spec: dict, train_dataset: Dataset, val_dataset: Dataset, cfg: dict) -> None:
    adapter_dir         = model_spec["adapter_dir"]
    model_name          = model_spec["name"]
    adapter_config_path = adapter_dir / "adapter_config.json"

    if adapter_config_path.exists():
        log.info(f"[{model_name}] Adapter already exists at {adapter_dir} — SKIPPING.")
        return

    adapter_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*65}")
    print(f"  Training: {model_name}  ({model_spec['model_id']})")
    print(f"{'='*65}")

    model = tokenizer = trainer = lora_config = bnb_config = training_args = None
    train_tok = val_tok = data_collator = None
    try:
        log.info(f"[{model_name}] Loading quantized model ...")
        bnb_config       = build_bnb_config(model_spec["compute_dtype"])
        model, tokenizer = load_model_and_tokenizer(model_spec, bnb_config)
        validate_token_lengths(pd.concat([train_dataset.to_pandas(), val_dataset.to_pandas()], ignore_index=True), tokenizer, model_spec, cfg)

        log.info(f"[{model_name}] Injecting LoRA adapters ...")
        lora_config = build_lora_config(cfg, model_spec)
        model       = apply_lora(model, lora_config)
        validate_lora_injection(model, model_spec)

        train_tok, val_tok = prepare_tokenized_datasets(train_dataset, val_dataset, tokenizer, model_spec, cfg)
        data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, label_pad_token_id=-100, pad_to_multiple_of=8)
        training_args = build_training_args(model_spec, adapter_dir, cfg)

        trainer = Trainer(
            model            = model,
            processing_class = tokenizer,
            args             = training_args,
            train_dataset    = train_tok,
            eval_dataset     = val_tok,
            data_collator    = data_collator,
        )

        trainer.add_callback(SafeEvalGenerationCallback(
            val_dataset, tokenizer, model_spec, cfg, max_eval_samples=cfg.get("VAL_GENERATION_N", 64)
        ))
        trainer.add_callback(ResourceAndStabilityCallback())
        trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))

        import datetime as dt
        config_snapshot = {
            "timestamp": dt.datetime.utcnow().isoformat(),
            "training_args": {
                "num_train_epochs": cfg["NUM_TRAIN_EPOCHS"],
                "learning_rate": cfg["LEARNING_RATE"],
                "lora_r": cfg["LORA_R"],
                "lora_alpha": cfg["LORA_ALPHA"],
                "lora_dropout": cfg["LORA_DROPOUT"],
                "per_device_batch_size": cfg["PER_DEVICE_BATCH_SIZE"],
                "gradient_accumulation": cfg["GRADIENT_ACCUMULATION"],
                "max_seq_length": cfg["MAX_SEQ_LENGTH"],
                "lr_scheduler": cfg["LR_SCHEDULER"],
                "warmup_ratio": cfg["WARMUP_RATIO"],
                "model_seed": MODEL_SEEDS.get(model_spec["model_id"], cfg["SEED"]),
            },
            "model_spec": {k: str(v) if isinstance(v, Path) else v for k, v in model_spec.items()},
            "dataset_config": {
                "train_samples": len(train_dataset),
                "val_samples": len(val_dataset),
                "aug_sample_ratio": cfg["AUG_SAMPLE_RATIO"],
                "n_folds": cfg["N_FOLDS"],
                "val_fold": cfg["VAL_FOLD"],
            }
        }
        with open(adapter_dir / "config_snapshot.json", "w") as f:
            json.dump(config_snapshot, f, indent=2, default=str)

        log.info(f"[{model_name}] Starting training ...")
        train_result = trainer.train()
        log.info(f"[{model_name}] Training complete — loss={train_result.training_loss:.4f}  steps={train_result.global_step}")

        log.info(f"[{model_name}] Saving LoRA adapter to {adapter_dir} ...")
        model.save_pretrained(str(adapter_dir))
        tokenizer.save_pretrained(str(adapter_dir))
        gen_metrics = run_validation_generation(model, tokenizer, model_spec, val_dataset, cfg)

        final_metrics = {
            "model_name":    model_name,
            "model_id":      model_spec["model_id"],
            "training_loss": train_result.training_loss,
            "global_step":   train_result.global_step,
            "train_samples": len(train_dataset),
            "val_samples":   len(val_dataset),
            **gen_metrics,
            "log_history_snapshot": trainer.state.log_history[-500:],
        }
        with open(adapter_dir / "training_metrics.json", "w") as f:
            json.dump(final_metrics, f, indent=2)
        print(f"  Adapter saved -> {adapter_dir}")

    finally:
        log.info(f"[{model_name}] Cleaning up VRAM ...")
        del trainer, model, tokenizer, train_tok, val_tok
        del data_collator, training_args, lora_config, bnb_config
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.synchronize()
            free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
            total_gb = torch.cuda.mem_get_info()[1] / 1024**3
            log.info(f"[{model_name}] VRAM after cleanup: {free_gb:.1f}/{total_gb:.1f} GB free")

print("Section 7 defined")


## Run — Group 1: Qwen3-1.7B + Qwen3-8B + LFM2.5-1.2B + Meta-Llama-3.1-8B-Instruct

```bash
nohup python run_phase2_group1.py > logs/phase2_group1.log 2>&1 &
tail -f logs/phase2_group1.log
```

Override GPU:
```bash
GPU_INDEX=2 python run_phase2_group1.py
```

Expected adapters:
```
adapters/qwen3_1_7b_adapter/
adapters/qwen3_8b_adapter/
adapters/lfm2_5_1_2b_adapter/
adapters/llama3_1_8b_adapter/
```

In [ ]:
def main():
    cfg = CONFIG
    set_seed(cfg["SEED"])

    import transformers
    from packaging.version import Version
    tf_ver = transformers.__version__
    if Version(tf_ver) < Version("5.5.0"):
        print(f"⚠ transformers=={tf_ver} — Gemma 4 requires >= 5.5.0.  Run: pip install -U transformers")
    else:
        print(f"✓ transformers=={tf_ver}")

    print("\n" + "="*65)
    print("  PHASE 2: SLM Ensemble QLoRA Training")
    print("="*65 + "\n")

    # Clear any residual VRAM before first model loads
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
        total_gb = torch.cuda.mem_get_info()[1] / 1024**3
        log.info(f"GPU ready: {free_gb:.1f}/{total_gb:.1f} GB free")

    print("▶ Step 1/3 — Loading and merging data ...")
    merged_df = load_and_merge_data(cfg)

    print("\n▶ Step 2/3 — GroupKFold split ...")
    train_dataset, val_dataset = make_train_val_datasets(merged_df, cfg)
    print(f"  Train: {len(train_dataset)} examples | Val: {len(val_dataset)} examples")

    cfg["ADAPTER_ROOT"].mkdir(parents=True, exist_ok=True)

    print(f"\n▶ Step 3/3 — Training {len(MODEL_REGISTRY)} models sequentially ...")
    for i, model_spec in enumerate(MODEL_REGISTRY):
        model_spec["adapter_dir"] = Path(model_spec["adapter_dir"]).resolve()
        print(f"\n  Model {i+1}/{len(MODEL_REGISTRY)}: {model_spec['name']}")
        train_one_model(model_spec, train_dataset, val_dataset, cfg)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print("\n" + "="*65)
    print("  Phase 2 complete — adapter summary:")
    adapter_root = cfg["ADAPTER_ROOT"].resolve()
    for m in MODEL_REGISTRY:
        exists = (Path(m["adapter_dir"]) / "adapter_config.json").exists()
        print(f"    {'✓' if exists else '✗'} {m['name']:25s} → {m['adapter_dir']}")
    print("="*65)
    print(f"\n  Adapters saved at: {adapter_root}")
    print("  Next step: upload adapters/ to Kaggle as a private dataset for offline inference.")

main()